# Normalize Raw Counts Matrices

## Purpose: 

Take raw counts matrices and normalize so that we have matrices for: 
* `TMM-normalized CPM` (with AND without `log2`)
* Gene-length normalized `geTMM` (with AND without `log2`)

This gives 4 total output datasets. 

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, conorm, numpy

In [2]:
cell_lines=["K562", "HepG2"]

gene_lengths=pd.read_csv(
    "../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/gene_lengths.tsv", 
    sep="\t", 
    index_col="Geneid"
)
gene_lengths.head()

,Length
Geneid,
DDX11L1,1735
WASH7P,1351
MIR6859-1,68
MIR1302-2HG,1021
MIR1302-2,138


In [3]:
for cell_line in cell_lines: 
    
    cell_line

    tmp_df = pd.read_csv(
        glob.glob("../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/{}*.tsv.gz".format(cell_line))[0],
        sep="\t", 
        compression="gzip",
        index_col="Geneid"
    )
    
    # add 1 count to avoid taking log of 0 
    tmp_df = tmp_df+1
        
    # get cpm counts using TMM norm factors
    tmm = conorm.cpm(tmp_df, norm_factors=conorm.tmm_norm_factors(tmp_df))
    
    # output as gzip'd TSV file
    tmm.to_csv(
        "../outputs/{}_tmm_no-log.tsv.gz".format(cell_line), 
        sep="\t", 
        compression="gzip", 
        index=True, 
        header=True
    )
    
    # same as above but log2 of all the CPM values 
    tmm = pd.DataFrame(numpy.log2(tmm))
    tmm.to_csv(
        "../outputs/{}_tmm_yes-log.tsv.gz".format(cell_line), 
        sep="\t", 
        compression="gzip", 
        index=True, 
        header=True
    )

    
    # get lengths of each gene in counts matrix 
    tmp_df = tmp_df.join(gene_lengths, how="outer", validate="1:1")
    # validate that each gene got a length value 
    assert tmp_df["Length"].notna().any()
    
    # run geTMM to get gene-length normalized 
    getmm = conorm.getmm(tmp_df, "Length")
    
    # output to file 
    getmm.to_csv(
        "../outputs/{}_getmm_no-log.tsv.gz".format(cell_line), 
        sep="\t", 
        compression="gzip", 
        index=True, 
        header=True
    )

    # same as above but log2 of all the CPM values 
    getmm = pd.DataFrame(numpy.log2(getmm))
    getmm.to_csv(
        "../outputs/{}_getmm_yes-log.tsv.gz".format(cell_line), 
        sep="\t", 
        compression="gzip", 
        index=True, 
        header=True
    )

'K562'

'HepG2'